In [ ]:
import pandas as pd
import numpy as np
from workflow.scripts.plotting_tools import get_model_colordict
from workflow.scripts.utils import get_forcing
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
order = [   
            
            'GISS-E2-1-G',
            'MIROC6',
            'GFDL-ESM4',
            'CNRM-ESM2-1',
            'UKESM1-0-LL',
            'IPSL-CM6A-LR-INCA',
            'NorESM2-LM',
            'MPI-ESM-1-2-HAM',
            'EC-Earth3-AerChem'
            
        ]

In [ ]:

def plot_forcings_bar(df, nmodels, pos0, ax,dist=.9, spacing_frac=.01, scaling: pd.Series=None
                      ,model_order: list = None):
    dx = dist/nmodels
    
    if scaling is not None:
        sign_diff = df['diff_sigificant'].copy()
        df = df.divide(scaling, axis=0)
        df['diff_sigificant'] = sign_diff
    if model_order:
        df = df.loc[order]
    else:
        df = df.sort_values('diff')
    
    pos=pos0
    gap =  spacing_frac/dist
    n=0
    for model, series in df.iterrows():
        if series.isnull()['diff']:
            continue
        else:
            if series['diff_sigificant'] == True:
                hatch='\\\\'
            else:
                hatch=None
            ax.barh(pos,series['diff'],height=dx-gap,zorder=100, facecolor=colors[model], 
                        xerr=series['pooled_std'],capsize=2, hatch=hatch)
            ax.plot(series['diff'],pos,  marker=".",  markerfacecolor=colors[model],
                    markeredgecolor= 'k',ms=10, zorder=300)
            
            if series['diff'] is not np.nan:
                n+=1
            pos+=dx
    mean = df.mean(axis=0)
    ax.plot(mean['diff'],pos0+dist/(n/2), mfc='#FF005E',linestyle='', 
                    marker='*', ms = 12, zorder=301, markeredgecolor='k')
        

In [ ]:
dfs = {p.split('.')[0].split('_')[-1]: pd.read_csv(p,index_col=0) for p in snakemake.input.forcing_tables}
colors = get_model_colordict()
vis_df = pd.read_csv(snakemake.input.diag_tables[0], index_col=0)
df_rel = pd.read_csv(snakemake.input.diag_tables[1], index_col=0)
ctrl_df = pd.read_csv(snakemake.input.diag_tables[2], index_col=0)
model_order = snakemake.params.get('model_order', order)

In [ ]:
vis_df = vis_df.rename(columns={'$\\Delta \\mathrm{Emiss}_{DU}$ \n (Tg/yr)': '$\\Delta \\mathrm{Emiss}_{DU}$ \n($\mathrm{Tg\;yr}^{-1}$)',
                               'DU MAC \n (m2 g-1)':'DU MAC \n ($\mathrm{m}^2\; \mathrm{g}^{-1}$)',
                               'DU MEC \n (m2 g-1)':'DU MEC \n ($\mathrm{m}^2\; \mathrm{g}^{-1}$)',
                                '$\Delta \mathrm{AAOD}_{550nm}$': '$(\dagger)\,\mathrm{DAOD}_{550}$',
                                '$\Delta \mathrm{AOD}_{550nm}$': '$(\dagger)\,\mathrm{DOD}_{550}$',
                                '$\mathrm{Angström}_{440-870}$'  : 'DU\n$\mathrm{Angstrom}_{440-870}$'
                               })

In [ ]:
def setup_plot_abs_Forcing(ax):
    ax.set_xlim(-1, 0.5)
    # ax.set_ylim(-0.25, 9.1)
    ax.xaxis.set_major_locator(mpl.ticker.FixedLocator([-0.8, -0.6, -0.4, -0.2, 0, 0.2,0.4]))
    ax.set_yticks([])
    ax.xaxis.set_minor_locator(mpl.ticker.AutoMinorLocator(4))
    ax.axvline(0.0, linestyle=":", linewidth=2, color="darkgrey")
    ax.tick_params(top=True, which="both", labeltop=True)
    ax.tick_params(which="major", labelsize=8)
    ax.spines["left"].set_visible(True)
    ax.spines["right"].set_visible(True)
    ax.set_xlabel("$\mathrm{W\; m}^{-2}$", fontsize=8)
    ax.set_title('Dust direct effective radiative forcing', fontsize=10)
    ax.text(0.98, 0.85, "LW", va="center", ha="right", transform=ax.transAxes, fontsize=8)
    ax.text(0.98, 0.5, "SW", va="center", ha="right", transform=ax.transAxes, fontsize=8)
    ax.text(0.98, 0.18, "Net", va="center", ha="right", transform=ax.transAxes, fontsize=8)
    return ax

In [ ]:
def _get_fmt(data):
    if abs(data) > 100:
        valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.0f}")
    elif abs(data) > 1:
        valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.1f}")
    elif abs(data) < 1 and abs(data) > 0.008:
        valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.3f}")
    else:
        valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.4f}")
    return valfmt_temp(data)
def annotate_heatmap(im,data, rel_change,valfmt="{x:.2f}", 
                     textcolors=["black", "white"], threshold=3, **textkw):
    """
    A function to annotate a heatmap.
    """
    # Normalize the threshold to the images color range.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",verticalalignment="center")
    kw.update(textkw)
    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = mpl.ticker.StrMethodFormatter(valfmt)
    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    cdata = im.get_array().data

    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(cdata[i, j] < threshold)])
            if np.isnan(data[i,j]):
                texts.append('')
            elif np.isnan(rel_change[i,j]):
                text = im.axes.text(j, i, f"{_get_fmt(data[i, j])}", **kw)
            else:
                text = im.axes.text(j, i, f"{_get_fmt(data[i, j])}\n ({_get_fmt(rel_change[i, j])} %)", **kw)
            texts.append(text)

    return texts


def setup_plot_forcingEff(ax):
    ax.set_xlim(-90, 30)
    # ax.set_ylim(-0.25, 9.1)
    ax.xaxis.set_major_locator(mpl.ticker.FixedLocator([-80,-60, -40, -20, 0, 20,40]))
    ax.set_yticks([])
    ax.xaxis.set_minor_locator(mpl.ticker.AutoMinorLocator(4))
    ax.axhline(3.9, linestyle="--", color="k")
    ax.tick_params(top=True, which="both", labeltop=True)
    ax.tick_params(which='major', labelsize=8)
    ax.spines["left"].set_visible(True)
    ax.spines["right"].set_visible(True)
    ax.text(
        0,
        0.725,
        "TOA",
        va="center",
        ha="right",
        rotation="vertical",
        transform=ax.transAxes,
        fontsize="8",
    )
    ax.text(
        0,
        0.22,
        "Surface",
        va="center",
        ha="right",
        rotation="vertical",
        transform=ax.transAxes,
        fontsize="8",
    )
    ax.axvline(0.0, linestyle=":", linewidth=2, color="darkgrey")
    ax.set_xlabel("$\mathrm{W\; m}^{-2} \;\mathrm{DOD}^{-1}$", labelpad=0.01, fontsize=8)
    ax.set_title('Dust direct effective radiative forcing per unit of DOD', fontsize=10)
    ax.text(0.98, 0.93, "LW", va="center", ha="right", transform=ax.transAxes, fontsize=8)
    ax.text(0.98, 0.725, "SW", va="center", ha="right", transform=ax.transAxes, fontsize=8)
    ax.text(0.98, 0.55, "Net", va="center", ha="right", transform=ax.transAxes, fontsize=8)
    ax.text(0.98, 0.32, "SW\nClearsky", va="center", ha="right", transform=ax.transAxes, fontsize=8)
    ax.text(0.98, 0.135, "Clearsky", va="center", ha="right", transform=ax.transAxes, fontsize=8)

    return ax

In [ ]:
rank_df = vis_df.rank(ascending=False)


In [ ]:
col_order = [
       'Lifetime \n (Days)',  'DU$_{Wetdep}$ \n /DU$_{Totdep}$', 
       'DU MEC \n ($\mathrm{m}^2\; \mathrm{g}^{-1}$)',
       'DU MAC \n ($\mathrm{m}^2\; \mathrm{g}^{-1}$)',
       'DU\n$\mathrm{Angstrom}_{440-870}$',
       '$\Delta \mathrm{Emiss}_{DU}$ \n($\mathrm{Tg\;yr}^{-1}$)',
       '$\Delta$DU burden \n (Tg)', 
       '$(\dagger)\,\mathrm{DOD}_{550}$',
       '$(\dagger)\,\mathrm{DAOD}_{550}$'
]

df_rel_col_order = ['lifetime','wetratio',
                    'od550dust_mass_ext',
                    'abs550aer_mass_abs',
                    'ang4487aer',
                    'emidust', 
                    'concdust_sum', 
                    'od550aer',
                    'abs550aer']

In [ ]:
vis_df = vis_df[col_order]
rank_df = rank_df[col_order]
df_rel = df_rel[df_rel_col_order]
aerChemMIP_mean = vis_df.mean(axis=0)
vis_df.loc['AerChemMIP mean'] = aerChemMIP_mean
aerChemMIP_mean_vis = vis_df.loc[['AerChemMIP mean']]
vis_df = vis_df.drop('AerChemMIP mean')

In [ ]:
new_ctrl_df = df_rel.copy()

In [ ]:
for c in df_rel_col_order:
    try:
        new_ctrl_df[c] = ctrl_df[c]
    except KeyError:
        new_ctrl_df[c] = np.nan

In [ ]:
mm_ctrl_df = new_ctrl_df.mean(axis=0)

In [ ]:
df_rel

In [ ]:
vis_df

In [ ]:
vis_df.columns

In [ ]:
extensive_mm = [       '$\Delta \mathrm{Emiss}_{DU}$ \n($\mathrm{Tg\;yr}^{-1}$)',
       '$\Delta$DU burden \n (Tg)', '$(\dagger)\,\mathrm{DOD}_{550}$',
       '$(\dagger)\,\mathrm{DAOD}_{550}$']
intensive_mm = ['Lifetime \n (Days)', 'DU$_{Wetdep}$ \n /DU$_{Totdep}$',
       'DU MEC \n ($\mathrm{m}^2\; \mathrm{g}^{-1}$)',
       'DU MAC \n ($\mathrm{m}^2\; \mathrm{g}^{-1}$)',
                'DU\n$\mathrm{Angstrom}_{440-870}$']


ctrl_extensive = ['emidust', 'concdust_sum', 'od550aer', 'abs550aer']
mean_vals_str = []
for d in intensive_mm:
    mval = _get_fmt(aerChemMIP_mean_vis[d].values[0])    
    mean_vals_str.append(f'{mval}')

for d,c in zip(extensive_mm,ctrl_extensive):
    rel_change = aerChemMIP_mean_vis[d].values/mm_ctrl_df[c]  
    rel_change = '({:.1f}%)'.format(rel_change[0]*100)
    mval = _get_fmt(aerChemMIP_mean_vis[d].values[0])    
    mean_vals_str.append(f'{mval}\n{rel_change}')



In [ ]:

context_dict = {
    "axes.labelsize": 10,
}
scaling_key = "$(\dagger)\,\mathrm{DOD}_{550}$"
with mpl.rc_context(context_dict):



    # Combined figure with GridSpec
    fig = plt.figure(figsize=(7.8, 9.7))  # Adjust the height to accommodate both subplots
    gs = fig.add_gridspec(3, 1, height_ratios=[8, 9, 1])  # Adjust height ratios to allocate space

    # First figure setup
    ax2 = fig.add_subplot(gs[0])
    axes = ax2  # Since the setup_plot function returns a single Axes
    setup_plot_forcingEff(axes)
    dist = 1.8
    plot_forcings_bar(
        get_forcing("ERFsurfcs", dfs),
        len(dfs),
        0.01,
        axes,
        dist=dist,
        scaling=vis_df[scaling_key],
        model_order=model_order,
    )
    plot_forcings_bar(
        get_forcing("ERFsurfswcs", dfs),
        len(dfs),
        2,
        axes,
        dist=dist,
        scaling=vis_df[scaling_key],
        model_order=model_order,
    )
    plot_forcings_bar(
        get_forcing("DirectEff", dfs),
        len(dfs),
        4.1,
        axes,
        dist=dist,
        scaling=vis_df[scaling_key],
        model_order=model_order,
    )
    plot_forcings_bar(
        get_forcing("SWDirectEff", dfs),
        len(dfs),
        5.8,
        axes,
        dist=dist,
        scaling=vis_df[scaling_key],
        model_order=model_order,
    )
    plot_forcings_bar(
        get_forcing("LWDirectEff", dfs),
        len(dfs),
        7.6,
        axes,
        dist=dist,
        scaling=vis_df[scaling_key],
        model_order=model_order,
    )

    legments = [
        Line2D([0], [0], markerfacecolor=colors[m], marker="o", label=m, color="w", markersize=10) for m in order[::-1]
    ]

    legments.append(
        Line2D(
            [0],
            [0],
            markerfacecolor="#FF005E",
            marker="*",
            label="Model mean",
            color="w",
            markeredgecolor="k",
            markersize=12,
        )
    )

    fig.legend(handles=legments, ncol=2, bbox_to_anchor=[0.05, 0.56, 0.5, 0.5], loc="lower left", 
               fontsize=8, frameon=False)

    # Second figure setup
    ax3 = fig.add_subplot(gs[1], aspect="equal", frameon=False)

    ax3.grid(color="w", linestyle="-", linewidth=3, which="minor")
    ax3.set_xticks(np.arange(vis_df.shape[1] + 1) - 0.5, minor=True)
    ax3.set_yticks(np.arange(vis_df.shape[0] + 1) - 0.5, minor=True)

    cmap = mpl.colormaps.get_cmap("YlGn_r").resampled(9)
    cmap.set_bad("#E6E6E6")
    im = ax3.imshow(rank_df, cmap=cmap, vmin=1, vmax=10, aspect="auto")
    cbar = ax3.figure.colorbar(im, ax=ax3, location="right", pad=0.02)
    cbar.ax.invert_yaxis()
    cbar.ax.set_yticks([2, 3, 4, 5, 6, 7, 8, 9, 10])
    cbar.ax.set_yticklabels(["1", "2", "3", "4", "5", "6", "7", "8", "9"])
    cbar.ax.set_title("Rank", fontsize=7.5)
    cbar.ax.set_position([0.92, 0.17, 0.03, 0.318])

    ax3.set_xticks(np.arange(vis_df.shape[1]), labels=vis_df.columns)
    ax3.xaxis.tick_top()
    ax3.set_yticks(np.arange(vis_df.shape[0]), labels=vis_df.index)
    ax3.tick_params(which="minor", bottom=False, left=False, top=False)
    ax3.tick_params(axis='x', labelsize=7)  # Change `8` to your desired fontsize
    ax3.tick_params(axis='y', labelsize=7)  # Change `8` to your desired fontsize
    ax4 = fig.add_subplot(gs[2], frameon=False)

    texts = annotate_heatmap(im, data=vis_df.values, rel_change=df_rel.values, threshold=4, fontsize=7.5)
    tab = ax4.table(
        cellText=[mean_vals_str],
        edges="horizontal",
        loc="top",
        bbox=[0, 0.4, 1.015, 1],
        cellLoc="center",
        rowLabels=["$Ens_{mean}$  −"],
        colLoc='right',
        
        

    )
    tab.auto_set_font_size(False)
    tab.set_fontsize(7.5)    
    # tab.auto_set_column_width(False)
    # tab.scale(xscale=2,yscale=1.7)
    celld = tab.get_celld()
    cello = celld[0, -1]
    cello.visible_edges = "open"
    cello.set_width(0.1)
    ax4.axis("off")

    
    pos2 = ax2.get_position()  # position of ax2
    pos3 = ax3.get_position()  # position of ax3
    pos4 = ax4.get_position()
    ax3.set_position([pos3.x0-0.03, pos3.y0 - 0.025, pos3.width+0.17, pos3.height-0.02])  # move ax3 down by 0.1 (adjust as needed)
    ax2.set_position([pos2.x0+0.36, pos2.y0, pos2.width-0.3, pos2.height+0.01]) 
    ax4.set_position([pos4.x0-0.026, pos4.y0, pos4.width+0.02, pos4.height])
    ax3.text(-0.1, 1.08, 'c)', transform=ax3.transAxes, fontsize=10, fontweight='bold', va='top', ha='right')
    ax2.text(0.0, 1.08, 'b)', transform=ax2.transAxes, fontsize=10, fontweight='bold', va='top', ha='right')
    ax1 = fig.add_axes([pos2.x0-0.09 + 0.02, pos2.y0+0.12, 0.38, pos2.height-0.11])  # adjust as needed
    ax1.text(-0.02, 1.1, 'a)', transform=ax1.transAxes, fontsize=10, fontweight='bold', va='top', ha='right')
    setup_plot_abs_Forcing(ax1)
    plot_forcings_bar(get_forcing('DirectEff', dfs),len(dfs), 0.02,ax1,dist=dist, model_order=model_order)
    plot_forcings_bar(get_forcing('SWDirectEff', dfs),len(dfs), 2,ax1,dist=dist, model_order=model_order)
    plot_forcings_bar(get_forcing('LWDirectEff', dfs),len(dfs), 4.1,ax1,dist=dist, model_order=model_order)
    fig.text(0.1,0.1,'($\dagger$) In parentheses, the percentage of total pre-industrial AOD (AAOD) that is due DOD (DAOD).',fontsize=7)

    plt.savefig(snakemake.output[0], bbox_inches="tight",dpi=300)
    
    plt.show()


In [ ]:
sum([0.09, 0.116, 0.116, 0.116, 0.116, 0.116, 0.116, 0.116, 0.116])

In [ ]:
[0.115 for i in range(9)]